# Second-pass model check on Colab

Select a **T4 GPU** runtime. This notebook reuses the saved first-pass JSON and tests Minerva perplexity plus Qwen entity review on **eight automatically sampled passages, two per collection**. This is a technical check, not a full-corpus LLM run or a final accuracy evaluation.

Before connecting, obtain access to `sapienzanlp/Minerva-350M-base-v1.0` on Hugging Face. Add a permitted read token in Colab's Secrets panel under `HF_TOKEN` and enable notebook access. Never put the token in a cell or commit it.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

repo = Path("/content/forensic-llm-graph")
if repo.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
print("Python:", sys.version.split()[0])


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
import json
from google.colab import userdata
from huggingface_hub import hf_hub_download

# The environment passes authentication to the child process without writing credentials.
token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("Add the HF_TOKEN Secret and enable notebook access.")
os.environ["HF_TOKEN"] = token
del token
settings = json.loads(Path("configs/hybrid.json").read_text(encoding="utf-8"))
hf_hub_download(settings["perplexity"]["model"], "config.json",
                revision=settings["perplexity"]["revision"])
print("Minerva access verified. No model weights loaded yet.")


## Upload the existing result

Using Colab's Files panel, upload `05_batched_form_boundaries_spacy_lg.json` to `/content/`. The upload is temporary and stays outside Git. No Drive mount or raw-data download is needed. Change `input_path` below if you chose another directory.


In [ ]:
input_path = Path("/content/05_batched_form_boundaries_spacy_lg.json")
if not input_path.is_file():
    raise FileNotFoundError("Upload run 05 using the Files panel before continuing.")
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
preview_path = Path("outputs/06_hybrid_preview.json")
subprocess.run([sys.executable, "scripts/run_hybrid.py", "--input", str(input_path),
                "--prepare-only", "--smoke-test", "--output", str(preview_path), "--overwrite"], check=True)
preview = json.loads(preview_path.read_text(encoding="utf-8"))
print("Input documents:", preview["run"]["input_document_count"])
print("Full-corpus passages:", preview["run"]["full_passage_count"])
print("Passages in this technical test:", preview["run"]["passages_by_source"])
assert preview["run"]["passages_by_source"] == {
    "redit": 2, "archivio_antimafia": 2, "dice": 2, "rizzo_synthetic": 2
}
del preview


## Run both models

Minerva loads first and scores the sampled passages, then it is released before Qwen loads. The first Qwen download contains approximately 16.4 GB of source weights even though GPU loading uses four-bit quantization. The model is downloaded once per fresh runtime.

Progress and model-loading messages appear below. Completed records are saved progressively. If interrupted, add `--resume` to the same command; a completed final JSON remains protected. The output's `scope` explicitly records `smoke_test`.


In [ ]:
output_path = Path("outputs/06_hybrid_smoke_test.json")
command = [sys.executable, "-u", "scripts/run_hybrid.py", "--input", str(input_path),
           "--smoke-test", "--output", str(output_path)]
# To continue an interrupted run in this runtime: command.append("--resume")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


In [ ]:
results = json.loads(output_path.read_text(encoding="utf-8"))
run = results["run"]
print("Scope:", run["scope"])
print("Input documents:", run["input_document_count"])
print("Represented documents:", run["document_count"])
print("Reviewed passages:", run["reviewed_passage_count"])
print("Perplexity status:", run["perplexity_status_counts"])
print("LLM status:", run["review_status_counts"])
print("GPU:", run["device"])
print("Elapsed seconds in this invocation:", round(run["invocation_seconds"], 2))
for role, metadata in run["models"].items():
    print(role, metadata)
if run["peak_host_memory_bytes"] is not None:
    print("Peak process RAM (GiB):", round(run["peak_host_memory_bytes"] / 2**30, 2))
for passage in results["passages"]:
    print("\n", passage["id"])
    print("Perplexity:", passage["perplexity"]["perplexity"])
    print("Review status:", passage["review"]["status"])
    print("Entities:", [(entity["text"], entity["label"]) for entity in passage["review"]["entities"]])


## Save and disconnect

Download `outputs/06_hybrid_smoke_test.json` manually for inspection. The `.checkpoints.jsonl` file is needed only to resume this run if the final result has not been produced; save it manually before deleting the runtime when necessary. Once the result is saved, disconnect while analysing it or making local changes.

A valid JSON response is not proof of correct NER. This test checks access, memory, perplexity computation, model generation and integration. Extended experiments still require development/test separation, a checked reference and calibrated routing settings.
